# **Spending, Income and LAYS Analysis**


# **Load Packages**

In [2]:
#========================================
# Import required packages
#========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Ellipse
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

from pathlib import Path
import json

from use_funcs import *
from use_funcs import Colors as C

# **Create Helper Functions**

Define plotting palette and project path helpers used throughout the notebook.

In [36]:
sc_theme()
sc_plotly_theme()

ig_colors = {
        "Low income": C.green,
        "Lower middle income": C.red,
        "Upper middle income": C.purple,
        "High income": C.blue,
    }

FileNotFoundError: Could not find path_config.json above c:\Users\F.Turner\Documents\00. Analyses\Random Notebooks

# **Load Data**

Load the prepared merged datasets used in the modeling and visualization steps.

In [5]:
#========================================
# Load core analysis datasets
#========================================

all_data = pd.read_csv("C:\\Users\\F.Turner\\Documents\\00. Analyses\\Education Financing\\Data\\all_data.csv")
lays_panels = pd.read_csv("C:\\Users\\F.Turner\\Documents\\00. Analyses\\Education Financing\\Data\\lays_panels.csv")

# **Analysis**

The following cells estimate the relationship between GDP per capita and LAYS, test alternative curve shapes, and assess differences by income group.

## Question 1: Does GDPPC Correlate with LAYS?

In [6]:
#========================================
# Build country-level base dataset for GDPPC-LAYS models
#========================================

df = all_data.copy()
df = df.dropna(subset=["lays", "gdppc_2015_usd"])
df = df.drop(df[df['income_group'] == 'Not classified'].index, axis=0)
df = df[["iso3",
         "year",
         "income_group",
         "pop_total",
         "lays",
         "gdppc_2015_usd",
         'expenditure_perchild_ppp',
         'expenditure_pctbudget_uis',
        'expenditure_pctgdp'
         ]]

df = df.groupby(["iso3"]).agg({
         "income_group": "first",
         "pop_total": "mean",
         "lays": "mean",
         "gdppc_2015_usd": "mean",
         'expenditure_perchild_ppp': "mean",
         'expenditure_pctbudget_uis': "mean",
        'expenditure_pctgdp': "mean"}).reset_index()

# What is the relationship between GDPPC and LAYS?

In [1]:

max_marker_size = 40
sizeref = 2.0 * df["pop_total"].max() / (max_marker_size**2)

fig = px.scatter(
    df,
    x="gdppc_2015_usd",
    y="lays",
    color="income_group",
    color_discrete_map=ig_colors,
    size="pop_total",
    log_x=True,
    hover_name="iso3",
    hover_data={
        "iso3": False,           # don't repeat it below
        "gdppc_2015_usd": ":,.0f",
        "lays": ":.1f",
        "pop_total": False,
        "income_group": False
    },
    labels={
        "gdppc_2015_usd": "GDPPC",
        "lays": "LAYS"
    }
)

fig.update_traces(marker=dict(sizemode='area', sizemin=3, sizeref=sizeref))

fig.update_layout(
        title={
        "text": "Learning Adjusted Years of Schooling vs GDP per Capita",
        "x": 0.5,           # centre
        "xanchor": "center",
        "font": {
            "size": 32,
            "family": "Oswald",
            "color": "black"
        }
    },
    xaxis_title="GDP per Capita (2015 USD) - Log Scale",
    yaxis_title="LAYS",
    height=800,
    margin=dict(l=100, r=100, t=100, b=100)
)

fig.update_xaxes(
    tickvals=[100, 1000, 10000, 100000],
)

fig.update_legends(title="Income Group")
fig.show()

fig.savefig("fig1_fullscatter.html")

NameError: name 'df' is not defined

## Do Slopes Vary by Income Group

In [133]:
#========================================
# Estimate interaction model across income groups
#========================================

df2 = df.copy()
df2 = df2.rename(columns={"gdppc_2015_usd": "gdppc", 'expenditure_perchild_ppp': "pcppp"})
df2['log_gdppc'] = np.log(df2['gdppc'])


# Create income group dummies
dummies = pd.get_dummies(df2["income_group"], prefix="inc", drop_first=True)
df2 = pd.concat([df2, dummies], axis=1)


In [132]:
#========================================
# Fit and plot separate log-GDPPC slopes 
# by income group
#========================================

## MODEL 5: Models per Income Group

df_lic = df2[df2['income_group'] == 'Low income']
df_lmic = df2[df2['income_group'] == 'Lower middle income']
df_umic = df2[df2['income_group'] == 'Upper middle income']
df_hic = df2[df2['income_group'] == 'High income']



In [65]:

# Scatter points
fig = px.scatter(
    df2,
    x="log_gdppc",
    y="lays",
    color="income_group",
    color_discrete_map=ig_colors,
    hover_name="iso3",
    hover_data={
        "iso3": False,
        "log_gdppc": False,
        "lays": ":.1f",
        "income_group": False
    },
    labels={
        "log_gdppc": "GDPPC",
        "lays": "LAYS"
    },
    opacity=0.7
)

groups = [
    (df_lic,  "Low Income",          C.green),
    (df_lmic, "Lower-middle Income", C.red),
    (df_umic, "Upper-middle Income", C.purple),
    (df_hic,  "High Income",         C.blue),
]

for df_grp, label, color in groups:

    X = sm.add_constant(df_grp["log_gdppc"])
    y = df_grp["lays"]

    model = sm.OLS(y, X).fit()

    x_line = np.linspace(
        df_grp["log_gdppc"].min(),
        df_grp["log_gdppc"].max(),
        100
    )

    y_line = model.predict(
        sm.add_constant(x_line)
    )

    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name=label,
            line=dict(
                color=color,
                width=2
            ),
            showlegend=False
        )
    )

    fig.update_legends(title_text="Income Group")

    fig.update_layout(
        template="sc",
        title={
            "text": "LAYS vs Log GDP Per Capita by Income Group",
            "x": 0.5,
            "xanchor": "center",
            "font": {
                "size": 32,
                "family": "Oswald",
                "color": "black"
            }
        },
        xaxis_title="Log of GDP per Capita (2015 USD)",
        yaxis_title="LAYS",
        height=800,
        margin=dict(l=100, r=100, t=100, b=100)
    
    )

fig.show()

## Question 2: Proportionate Expenditure and LAYS


In [110]:
fig = px.scatter(data_frame=df, x="expenditure_pctgdp", y="lays",
                 color="income_group", 
                 color_discrete_map=ig_colors,
                 hover_name="iso3",
                 hover_data={"expenditure_pctgdp": ":.0%", 
                             "lays": ":.1f",
                             "iso3": False,
                             "income_group": False},
                 labels={"expenditure_pctgdp": "Expenditure as % of GDP", "lays": "LAYS"},
                 template="sc")

fig.update_layout(
    title={"text": "LAYS vs Expenditure as % of GDP",
           "x":0.5,
           "font": {"family": "Oswald",
                    "size": 32},
           "xanchor": "center"
           },
    height=600,
    width=900,
    
)

fig.update_legends(title="Income Group")

fig.add_shape(
    type="line",
    x0=0.06,
    x1=0.06,
    y0=0,
    y1=14,
    line=dict(color=C.grey_500, width=1)
)

fig.add_shape(
    type="line",
    x0=0,
    x1=.16,
    y0=df["lays"].median(),
    y1=df["lays"].median(),
    line=dict(color=C.grey_500, width=1)
)

fig.add_annotation(
    x=0.08,
    y=0.3,
    text="6% of GDP Benchmark",
    showarrow=False,
    align="left"
)

fig.add_annotation(
    x=0.13,
    y=8.4,
    text="Global Median LAYS",
    showarrow=False,
    align="left"
)

fig.show()

## **Change in LAYS and Budget over Time**

This uses annualised panel data from lays_panels. This captures figures for baseline (earliest year in panel) and delta_ (change across a panel annualised). There are multiple panels covering different time period, but when aggregated we take the average annualised change across panels for a country. 


In [113]:

#=====================================================================
# Prepare panel dataset for plotting change in education expenditure vs change in LAYS
#=====================================================================
df3 = lays_panels.copy()
df3 = lays_panels[lays_panels['panel'].isin([1, 2, 3, 4, 5])]

agg_map = {col: 'mean' for col in df3.columns if col not in ['iso3', 'income_group']}
agg_map['income_group'] = 'first'
df3 = df3.groupby('iso3', as_index=False).agg(agg_map)

df3 = df3[df3['income_group'].isin(['Low income', 'Lower middle income', 'Upper middle income', 'High income'])]


C:\Users\F.Turner\AppData\Local\Temp\ipykernel_13528\2051466054.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df3 = df3.groupby('iso3', as_index=False).agg(agg_map)


In [139]:
df3.columns.to_list()

['iso3',
 'index',
 'start_year',
 'end_year',
 'baseline_lays',
 'delta_lays',
 'baseline_lays_male',
 'delta_lays_male',
 'baseline_lays_female',
 'delta_lays_female',
 'baseline_pcr_modelled_pct',
 'delta_pcr_modelled_pct',
 'baseline_pcr_pct',
 'delta_pcr_pct',
 'baseline_pcr_wpia',
 'delta_pcr_wpia',
 'baseline_general_public_services_pct',
 'delta_general_public_services_pct',
 'baseline_defense_pct',
 'delta_defense_pct',
 'baseline_public_order_safety_pct',
 'delta_public_order_safety_pct',
 'baseline_economic_affairs_pct',
 'delta_economic_affairs_pct',
 'baseline_environmental_protection_pct',
 'delta_environmental_protection_pct',
 'baseline_housing_community_amenities_pct',
 'delta_housing_community_amenities_pct',
 'baseline_health_pct',
 'delta_health_pct',
 'baseline_recreation_culture_religion_pct',
 'delta_recreation_culture_religion_pct',
 'baseline_education_pct',
 'delta_education_pct',
 'baseline_social_protection_pct',
 'delta_social_protection_pct',
 'baseline_se

In [156]:
max_marker_size = 40
sizeref = 2.0 * df3["baseline_pop_total"].max() / (max_marker_size**2)

fig = px.scatter(data_frame=df3, x="delta_expenditure_pctgdp", y="delta_lays",
                 color="income_group",
                 color_discrete_map=ig_colors,
                 hover_name="iso3",
                 hover_data={"delta_expenditure_pctgdp": ":.2%", 
                             "delta_lays": ":.2f",
                             "iso3": False,
                             "income_group": False, 
                             "baseline_pop_total": False},
                 labels={"delta_expenditure_pctgdp": "Change in Education Expenditure as % of GDP", "delta_lays": "Annualised Change in LAYS"},
                 template="sc",
                 size="baseline_pop_total",
                 size_max=max_marker_size,
                 )

fig.update_traces(marker=dict(sizemode='area', sizeref=sizeref,  sizemin=1))

fig.update_layout(
    title={"text": "Annualised Change in LAYS vs Change in Education Expenditure as % of GDP",
           "x":0.5,
           "font": {"family": "Oswald",
                    "size": 32},
           "xanchor": "center"
           },
    height=600,
    width=900,
    hoverlabel=dict(
    bgcolor=C.grey_100,
    font_size=10,
    font_family="Arial"
    )
    
)



fig.update_xaxes(title_text="Change in Education Expenditure as % of GDP", range=[-0.01, 0.01], showgrid=True, gridwidth=1)
fig.update_yaxes(title_text="Annualised Change in LAYS", range=[-0.4, 0.4], showgrid=True, gridwidth=1)

fig.update_legends(title="Income Group")

fig.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=-0.43,
    y1=0.56,
    line=dict(color=C.grey_500, width=1)
)

fig.add_shape(
    type="line",
    x0=-0.021,
    x1=0.02,
    y0=0,
    y1=0,
    line=dict(color=C.grey_500, width=1)
)

fig.add_shape(
    type="rect",
    x0=-0.021,
    x1=0.02,
    y0=-.43,
    y1=0,
    fillcolor=C.medium_red,
    opacity=0.3,
    line=dict(color="rgba(0,0,0,0)"),
    layer="below"
)

fig.add_shape(
    type="rect",
    x0=-0.021,
    x1=0.02,
    y0=0,
    y1=0.56,
    fillcolor=C.medium_green,
    opacity=0.3,
    line=dict(color="rgba(0,0,0,0)"),
    layer="below"
)

fig.show()